# 02. Model Estimation

## Set Up

In [42]:
import pythonnet
pythonnet.load("coreclr")

import clr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from helper_functions import resolve_bestfit_dll, resolve_numerics_dll, convert_to_dotnet_array

clr.AddReference(str(resolve_numerics_dll()))
clr.AddReference(str(resolve_bestfit_dll()))

from RMC.BestFit import ExactData, ExactSeries
from RMC.BestFit.Estimation import MaximumLikelihood, OptimizationMethod, BayesianAnalysis
from RMC.BestFit.Models import DataFrame, UnivariateDistribution

from Numerics.Mathematics.Optimization import ParameterSet
from Numerics.Distributions import UnivariateDistributionType

from System import Double, Random 

Parameter estimation is the process of finding the values that make a model best explain observed data. BestFit provides full uncertainty quantification through Markov Chain Monte Carlo (MCMC) sampling. It also supports Maximum Likelihood Estimation (MLE) for point estimates and the Generalized Method of Moments (GMM) for specific applications.

## Why Estimation Method Matters

Consider fitting a Generalized Extreme Value (GEV) distribution to 50 years of annual peak flows. Different estimation methods give different information:

| Method | Output | Use Case |
|--------|--------|----------|
| **MLE** | Point estimates, standard errors | Quick analysis, operational forecasting |
| **Bayesian MCMC** | Full posterior distributions | Uncertainty quantification, decision-making under uncertainty |
| **GMM** | Point estimates, moment conditions | Robust to distributional assumptions |
| **MAP** | Posterior mode | Regularized point estimate with prior information |

For life-safety applications like dam and levee risk assessment, **Bayesian MCMC is the recommended approach** because it properly propagates parameter uncertainty to design quantiles and risk estimates.

---

## Maximum Likelihood Estimation (MLE)

### Theoretical Foundation

Maximum Likelihood Estimation finds the parameter values $\hat{\theta}$ that maximize the probability of observing the data:

```math
\hat{\theta}_{\text{MLE}} = \arg\max_{\theta} \mathcal{L}(\theta | \mathbf{y}) = \arg\max_{\theta} \prod_{i=1}^{n} f(y_i | \theta)
```

Equivalently, we maximize the log-likelihood:

```math
\hat{\theta}_{\text{MLE}} = \arg\max_{\theta} \ell(\theta) = \arg\max_{\theta} \sum_{i=1}^{n} \log f(y_i | \theta)
```

### Properties of MLE

Under regularity conditions, MLE has desirable asymptotic properties:

1. **Consistency**: $\hat{\theta}_{\text{MLE}} \xrightarrow{p} \theta_0$ as $n \to \infty$
2. **Asymptotic Normality**: $\sqrt{n}(\hat{\theta}_{\text{MLE}} - \theta_0) \xrightarrow{d} N(0, I^{-1}(\theta_0))$
3. **Efficiency**: Achieves the Cramér-Rao lower bound asymptotically

where $I(\theta)$ is the Fisher Information matrix:

```math
I(\theta) = -E\left[\frac{\partial^2 \ell(\theta)}{\partial \theta \partial \theta^T}\right]
```

### Implementation in RMC-BestFit

The `MaximumLikelihood` class provides MLE for any model implementing `IModel`:


In [ ]:
annual_peaks = [
    45000, 52000, 38000, 61000, 49000, 55000, 42000, 67000, 39000, 48000,
    51000, 36000, 58000, 44000, 53000, 47000, 62000, 41000, 50000, 37000,
    54000, 46000, 59000, 43000, 56000, 40000, 63000, 35000, 57000, 45000
]

df = DataFrame()
df.ExactSeries = ExactSeries(convert_to_dotnet_array(annual_peaks))

model = UnivariateDistribution(df, UnivariateDistributionType.GeneralizedExtremeValue)
mle = MaximumLikelihood(model)

# Choose optimization method
mle.Method = OptimizationMethod.NelderMead;           # Local optimizer (fast)
# mle.Method = OptimizationMethod.MultilevelSingleLinkage;  # Global optimizer (robust)
# mle.Method = OptimizationMethod.DifferentialEvolution;    # Global optimizer (very robust)

mle.Estimate()

if mle.IsEstimated:
    print("MLE Results")
    print(f"Log Likelihood: {mle.BestParameterSet.Fitness:.2f}")

    for i in range(len(model.Parameters)):
        print(f" {model.Parameters[i].DisplayName} : {mle.BestParameterSet.Values[i]:.2f}")

    model.SetParameterValues(mle.BestParameterSet.Values)

MLE Results
Log Likelihood: 314.07
 Location (ξ) : 45860.27
 Scale (α) : 8223.39
 Shape (κ) : 0.24


### Optimization Methods
BestFit provides several optimization algorithms through the Numerics library:

| Method | Type | Speed | Robustness | Recommended For |
|--------|------|-------|------------|-----------------|
| `NelderMead` | Local | Fast | Low | Well-behaved likelihoods |
| `BFGS` | Local | Fast | Medium | Smooth likelihoods |
| `Powell` | Local | Medium | Medium | Non-differentiable objectives |
| `DifferentialEvolution` | Global | Slow | High | Multimodal likelihoods |
| `MultilevelSingleLinkage` | Global | Medium | High | General use |
| `ParticleSwarm` | Global | Medium | Medium | High-dimensional problems |

**Recommendation**: Use `MultilevelSingleLinkage` for most applications. It balances computational cost with robustness to local optima.

### Handling Multiple Local Optima

Many hydrologic models have multimodal likelihoods. For example, rating curves with the power-law form $Q = \alpha(h - \xi)^\beta$ have parameter trade-offs: different combinations of $(\xi, \alpha, \beta)$ can produce similar fits.

To increase confidence in finding the global optimum:

In [ ]:
# Run multiple optimization from different starting points
best_fitness = Double.NegativeInfinity
best_params = ParameterSet()

for trial in range(10):
    mle = MaximumLikelihood(model, OptimizationMethod.NelderMead)

    # Random starting point within bounds
    random = Random(trial)
    start_values = np.zeros(model.Parameters.Count)
    for i in range(model.Parameters.Count):
        p = model.Parameters[i]
        start_values[i] = p.LowerBound + random.NextDouble() * (p.UpperBound - p.LowerBound)
    mle.InitialValues = start_values

    mle.Estimate()

    if mle.IsEstimated and mle.BestParameterSet.Fitness > best_fitness:
        best_fitness = mle.BestParameterSet.Fitness
        best_params = mle.BestParameterSet


print(f"Best log likelihood across trials: {best_fitness:.2f}")
print("Best parameters:")
for i in range(model.Parameters.Count):
    print(f" {model.Parameters[i].DisplayName} : {best_params.Values[i]:.2f}")

Best log likelihood across trials: 314.07
Best parameters:
 Location (ξ) : 45860.27
 Scale (α) : 8223.39
 Shape (κ) : 0.24


## Bayesian MCMC Estimation

### Theoretical Foundation

Bayesian inference treats parameters as random variables with probability distributions. We start with a **prior distribution** $\pi(\theta)$ encoding our beliefs before seeing data, then update to a posterior distribution after observing data $\mathbf{y}$:

```math
\pi(\theta | \mathbf{y}) = \frac{\mathcal{L}(\mathbf{y} | \theta) \cdot \pi(\theta)}{\int \mathcal{L}(\mathbf{y} | \theta) \cdot \pi(\theta) \, d\theta}
```

This is Bayes' theorem. The denominator (marginal likelihood or evidence) is typically intractable, but MCMC methods sample from the posterior without computing it.

### Why Bayesian?

Bayesian estimation offers several advantages for hydrologic applications:

1. **Full Uncertainty Quantification**: Get probability distributions, not just point estimates
2. **Natural Framework for Prediction**: Predictive distributions integrate over parameter uncertainty
3. **Incorporation of Prior Information**: Engineering judgment can inform priors
4. **Coherent Decision-Making**: Posterior distributions support risk-based decisions
5. **Flexible Model Comparison**: WAIC, LOO-CV, and Bayes factors

### The DEMCzs Sampler

BestFit uses the Differential Evolution Markov Chain with snooker update (DEMCzs) algorithm [[1]](#1) as its primary MCMC sampler. DEMCzs is particularly well-suited for:

- **High-dimensional problems**: Efficient even with 10+ parameters
- **Correlated parameters**: Adapts proposal scale and direction automatically
- **Multimodal posteriors**: Multiple chains can explore different modes

The algorithm maintains $N$ chains (typically $N = 3$) that evolve in parallel. Each chain proposes new states using differences between other chains:

```math
\theta^* = \theta^{(i)} + \gamma \cdot (\theta^{(r_1)} - \theta^{(r_2)}) + \varepsilon
```

where $r_1, r_2$ are randomly selected chains and $\gamma$ is a scaling factor.

### Implementation in RMC-BestFit

The `BayesianAnalysis` class provides MCMC estimation:

In [61]:
df = DataFrame()
df.ExactSeries = ExactSeries(convert_to_dotnet_array(annual_peaks))
model = UnivariateDistribution(df, UnivariateDistributionType.GeneralizedExtremeValue)

# Configure Bayesian analysis
bayesian = BayesianAnalysis(model)

# Sampling parameters
bayesian.Type = BayesianAnalysis.SamplerType.DEMCzs  # Default sampler
bayesian.Iterations = 10000           # Total iterations after warmup
bayesian.WarmupIterations = 5000      # Burn-in period (discarded)
bayesian.ThinningInterval = 1         # Keep every nth sample (1 = no thinning)

# Run Bayesian estimation
bayesian.RunAsync().Wait()

results = bayesian.Results

print("Bayesian Analysis Results:")
print(F"{"Parameter":<20} {"Mean":>10} {"StdDev":>10} {"2.5%":>10} {"97.5%":>10} {"Rhat":>8} {"ESS":>8}")

for i in range(model.Parameters.Count):
    stats = results.ParameterResults[i].SummaryStatistics

    print(
    f"{model.Parameters[i].DisplayName:<20} "
    f"{stats.Mean:10.3f} "
    f"{stats.StandardDeviation:10.3f} "
    f"{stats.LowerCI:10.3f} "
    f"{stats.UpperCI:10.3f} "
    f"{stats.Rhat:8.3f} "
    f"{stats.ESS:8.3f}"
)

Bayesian Analysis Results:
Parameter                  Mean     StdDev       2.5%      97.5%     Rhat      ESS
Location (ξ)          45483.415   1790.746  42551.811  48498.134    1.002  708.431
Scale (α)              8690.016   1497.275   6699.840  11323.457    1.001  611.037
Shape (κ)                 0.173      0.177     -0.122      0.439    1.000  677.729
